In [1]:
import scanpy as sc
import numpy as np

In [2]:
# defining rank genes zones for tangram
# using all normal samples in RNA multiple
# downsampling for 10k cells out of all normal cells for tangram reference
# running rank genes on downsamples 10k cell reference

In [2]:
adata_normal = sc.read_h5ad('/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/spatial/spatial_scr/condition_h5ad/Control.h5ad')
sc.pp.normalize_total(adata_normal) #it seems like its raw counts
# Logarithmize the data
sc.pp.log1p(adata_normal)
adata_normal_heps = adata_normal[adata_normal.obs['celltype'].isin(['Hepatocytes'])]
adata_normal_heps.obs['donor_demux'].value_counts()

donor_demux
HL190215    9211
HL211001    8046
HL170062    7172
HL160040    7098
HL160026    6012
HL180809    5416
HL210621    3685
HL200115    2977
HL170044    2916
HL170052    2721
HL230316    2625
HL221119    2178
HL160037    1869
HL200805    1800
HL220402    1780
HL220613    1766
HL210207    1630
HL190624    1300
HL170047    1183
HL160029    1104
HL210614    1050
HL220210     981
HL211106     830
HL220825     597
HL190214     494
HL220216     410
Name: count, dtype: int64

In [3]:
adata_normal_heps.obs['cellsubtype'].value_counts()

cellsubtype
Hepatocytes.Zone2        29321
Hepatocytes.Zone3        26906
Hepatocytes.Zone1        14271
Hepatocytes.Senescent     6200
Hepatocytes.Stress         153
Name: count, dtype: int64

In [ ]:
# remove stress and senescent clusters

In [4]:
adata_normal_heps = adata_normal_heps[~adata_normal_heps.obs['cellsubtype'].isin(['Hepatocytes.Senescent', 'Hepatocytes.Stress'])]
adata_normal_heps.obs['cellsubtype'].value_counts()

cellsubtype
Hepatocytes.Zone2    29321
Hepatocytes.Zone3    26906
Hepatocytes.Zone1    14271
Name: count, dtype: int64

In [6]:
adata_normal_heps

AnnData object with n_obs × n_vars = 70498 × 36601
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'nCount_RNA_raw', 'nFeature_RNA_raw', 'donor_demux', 'nCount_SCT', 'nFeature_SCT', 'SCT.weight', 'seurat_clusters', 'log_nCount_SCT', 'log_nFeature_SCT', 'lane', 'batch', 'gex_raw_reads', 'gex_mapped_reads', 'gex_conf_intergenic_reads', 'gex_conf_exonic_reads', 'gex_conf_intronic_reads', 'gex_conf_exonic_unique_reads', 'gex_conf_exonic_antisense_reads', 'gex_conf_exonic_dup_reads', 'gex_exonic_umis', 'gex_conf_intronic_unique_reads', 'gex_conf_intronic_antisense_reads', 'gex_conf_intronic_dup_reads', 'gex_intronic_umis', 'gex_conf_txomic_unique_reads', 'gex_umis_count', 'gex_genes_count', 'atac_raw_reads', 'atac_unmapped_reads', 'atac_lowmapq', 'atac_dup_reads', 'atac_chimeric_reads', 'atac_mitochondrial_reads', 'atac_fragments', 'atac_TSS_fragments', 'atac_peak_region_fragments', 'atac_peak_region_cutsites', 'TSS.enrichment', 'TSS.percentile', 'condition', 'disease_sta

In [7]:
sc_cells = 10000

In [8]:
# User input: Set n_obs to number of cells to keep

sc.pp.subsample(adata_normal_heps, n_obs=sc_cells, random_state=42)


In [11]:
adata_normal_heps

AnnData object with n_obs × n_vars = 10000 × 36601
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'nCount_RNA_raw', 'nFeature_RNA_raw', 'donor_demux', 'nCount_SCT', 'nFeature_SCT', 'SCT.weight', 'seurat_clusters', 'log_nCount_SCT', 'log_nFeature_SCT', 'lane', 'batch', 'gex_raw_reads', 'gex_mapped_reads', 'gex_conf_intergenic_reads', 'gex_conf_exonic_reads', 'gex_conf_intronic_reads', 'gex_conf_exonic_unique_reads', 'gex_conf_exonic_antisense_reads', 'gex_conf_exonic_dup_reads', 'gex_exonic_umis', 'gex_conf_intronic_unique_reads', 'gex_conf_intronic_antisense_reads', 'gex_conf_intronic_dup_reads', 'gex_intronic_umis', 'gex_conf_txomic_unique_reads', 'gex_umis_count', 'gex_genes_count', 'atac_raw_reads', 'atac_unmapped_reads', 'atac_lowmapq', 'atac_dup_reads', 'atac_chimeric_reads', 'atac_mitochondrial_reads', 'atac_fragments', 'atac_TSS_fragments', 'atac_peak_region_fragments', 'atac_peak_region_cutsites', 'TSS.enrichment', 'TSS.percentile', 'condition', 'disease_sta

In [ ]:
# no stressed population and sen population

In [12]:
sc.tl.rank_genes_groups(adata_normal_heps, groupby="cellsubtype", use_raw=False)


In [13]:
import pandas as pd
import csv

In [14]:
markers_df_adata_normal_heps = pd.DataFrame(adata_normal_heps.uns["rank_genes_groups"]["names"]).iloc[:, :]


In [15]:
markers_df_adata_normal_heps

,Hepatocytes.Zone1,Hepatocytes.Zone2,Hepatocytes.Zone3
0,HAL,FGF14,SLCO1B3
1,GOT1,PTP4A1,SLCO1B7
2,PPARGC1A,ORM1,AC004053.1
3,SLC7A2,HAMP,CYP3A4
4,SDS,LINC02197,ADH4
...,...,...,...
36596,ADH4,PAH,TOX
36597,AC004053.1,AC004053.1,PTP4A1
36598,SLCO1B7,CYP3A4,AC007221.1
36599,SLCO1B3,SLCO1B7,FGF14


In [16]:
# Iterate through each column and save it to a text file
for celltype in markers_df_adata_normal_heps.columns:
    #celltype = 'Hepatocytes.Senescent'
        # Extract the genes for the current cell type
    genes = markers_df_adata_normal_heps[celltype].dropna()  # Drop any NaN values if present
    
    # Add quotes around each gene
    genes_with_quotes = [f'{gene}' for gene in genes]
    #print(genes.head())
    # Create a dataframe with column name "x"
    output_df = pd.DataFrame(genes_with_quotes, columns=["x"])
    
    
    print(output_df.head())
    # Save to a text file named after the cell type
    output_df.to_csv(f"/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/scripts/zone_gene_lists/Normal_alldonors_nosen_stress_{celltype}_RNA_Multiome_RankGenesALL.txt",
                     index=False, sep="\t",quoting=csv.QUOTE_ALL)

          x
0       HAL
1      GOT1
2  PPARGC1A
3    SLC7A2
4       SDS
           x
0      FGF14
1     PTP4A1
2       ORM1
3       HAMP
4  LINC02197
            x
0     SLCO1B3
1     SLCO1B7
2  AC004053.1
3      CYP3A4
4        ADH4


In [17]:

markers_df_adata_normal_heps_top50 = pd.DataFrame(adata_normal_heps.uns["rank_genes_groups"]["names"]).iloc[0:50, :]
markers_df_adata_normal_heps_top50.head()

,Hepatocytes.Zone1,Hepatocytes.Zone2,Hepatocytes.Zone3
0,HAL,FGF14,SLCO1B3
1,GOT1,PTP4A1,SLCO1B7
2,PPARGC1A,ORM1,AC004053.1
3,SLC7A2,HAMP,CYP3A4
4,SDS,LINC02197,ADH4


In [18]:
markers_df_adata_normal_heps_top50.shape # Check out dimensions

(50, 3)

In [17]:
#markers_df_adata_normal_heps_top50

In [19]:
markers_df_adata_normal_heps_top50 = list(np.unique(markers_df_adata_normal_heps_top50.melt().value.values))   # Create an array of the unique markers across all the cell types
len(markers_df_adata_normal_heps_top50)

150

In [19]:
#adata_normal_heps

In [20]:
adata_normal_heps.__dict__['_raw'].__dict__['_var'] = adata_normal_heps.__dict__['_raw'].__dict__['_var'].rename(columns={'_index': 'features'})

In [21]:
import os

In [22]:
gene_nums = 50

export_dir = '/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/outputs/sandbox/spatial/250214_tangram_preprocessing_genelist50_liver_normal_heps_alldonors_top50_nosen_stress/'

if not os.path.isdir(export_dir):
    os.mkdir(export_dir)

print(export_dir)

/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/outputs/sandbox/spatial/250214_tangram_preprocessing_genelist50_liver_normal_heps_alldonors_top50_nosen_stress/


In [24]:
# Write out the preprocessed objects to be fed into a python script that runs tangram

adata_normal_heps.write(export_dir + 'Preprocessed_adsc.h5ad')
#ad_sp.write(export_dir + 'Preprocessed_adsp.h5ad')

file = open(export_dir +'markers','w')
for item in markers_df_adata_normal_heps_top50:
	file.write(item+"\n")
file.close()